Optical Flow 
============

Goal
----

In this chapter,
    -   We will understand the concepts of optical flow and its estimation using Lucas-Kanade
        method.
    -   We will use functions like **cv.calcOpticalFlowPyrLK()** to track feature points in a
        video.
    -   We will create a dense optical flow field using the **cv.calcOpticalFlowFarneback()** method.

Optical Flow
------------

Optical flow is the pattern of apparent motion of image objects between two consecutive frames
caused by the movement of object or camera. It is 2D vector field where each vector is a
displacement vector showing the movement of points from first frame to second. Consider the image
below (Image Courtesy: [Wikipedia article on Optical Flow](http://en.wikipedia.org/wiki/Optical_flow)).

![image](images/optical_flow_basic1.jpg)

It shows a ball moving in 5 consecutive frames. The arrow shows its displacement vector. Optical
flow has many applications in areas like :

-   Structure from Motion
-   Video Compression
-   Video Stabilization ...

Optical flow works on several assumptions:

1. The pixel intensities of an object do not change between consecutive frames.
2. Neighbouring pixels have similar motion.

Consider a pixel $I(x,y,t)$ in first frame (Check a new dimension, time, is added here. Earlier we
were working with images only, so no need of time). It moves by distance $(dx,dy)$ in next frame
taken after $dt$ time. So since those pixels are the same and intensity does not change, we can say,

$$I(x,y,t) = I(x+dx, y+dy, t+dt)$$

Then take taylor series approximation of right-hand side, remove common terms and divide by $dt$ to
get the following equation:

$$f_x u + f_y v + f_t = 0$$

where:

$$f_x = \frac{\partial f}{\partial x}, \qquad f_y = \frac{\partial f}{\partial y}$$
$$u = \frac{dx}{dt}, \qquad v = \frac{dy}{dt}$$

Above equation is called Optical Flow equation. In it, we can find $f_x$ and $f_y$, they are image
gradients. Similarly $f_t$ is the gradient along time. But $(u,v)$ is unknown. We cannot solve this
one equation with two unknown variables. So several methods are provided to solve this problem and
one of them is Lucas-Kanade.

### Lucas-Kanade method

We have seen an assumption before, that all the neighbouring pixels will have similar motion.
Lucas-Kanade method takes a 3x3 patch around the point. So all the 9 points have the same motion. We
can find $(f_x, f_y, f_t)$ for these 9 points. So now our problem becomes solving 9 equations with
two unknown variables which is over-determined. A better solution is obtained with least square fit
method. Below is the final solution which is two equation-two unknown problem and solve to get the
solution.

$$\begin{bmatrix} u \\ v \end{bmatrix} =
\begin{bmatrix}
    \sum_{i}{f_{x_i}}^2  &  \sum_{i}{f_{x_i} f_{y_i} } \\
    \sum_{i}{f_{x_i} f_{y_i}} & \sum_{i}{f_{y_i}}^2
\end{bmatrix}^{-1}
\begin{bmatrix}
    - \sum_{i}{f_{x_i} f_{t_i}} \\
    - \sum_{i}{f_{y_i} f_{t_i}}
\end{bmatrix}$$

( Check similarity of inverse matrix with Harris corner detector. It denotes that corners are better
points to be tracked.)

So from the user point of view, the idea is simple, we give some points to track, we receive the optical
flow vectors of those points. But again there are some problems. Until now, we were dealing with
small motions, so it fails when there is a large motion. To deal with this we use pyramids. When we go up in
the pyramid, small motions are removed and large motions become small motions. So by applying
Lucas-Kanade there, we get optical flow along with the scale.

Lucas-Kanade Optical Flow in OpenCV
-----------------------------------

OpenCV provides all these in a single function, **cv.calcOpticalFlowPyrLK()**. Here, we create a
simple application which tracks some points in a video. To decide the points, we use
**cv.goodFeaturesToTrack()**. We take the first frame, detect some Shi-Tomasi corner points in it,
then we iteratively track those points using Lucas-Kanade optical flow. For the function
**cv.calcOpticalFlowPyrLK()** we pass the previous frame, previous points and next frame. It
returns next points along with some status numbers which has a value of 1 if next point is found,
else zero. We iteratively pass these next points as previous points in next step. See the code
below:

In [ ]:
from time import sleep

import cv2 as cv
import numpy as np
from ipycanvas import Canvas, hold_canvas
from IPython.display import display
from ipywidgets import Button

# Shared restart flag
_restart = False


def on_restart_clicked(b):
    global _restart
    _restart = True


# Restart button
restart_btn = Button(description="Restart")
restart_btn.on_click(on_restart_clicked)
display(restart_btn)

video = "../videos/slow_traffic_small.mp4"
cap = cv.VideoCapture(video)

# params for ShiTomasi corner detection
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

# Parameters for lucas kanade optical flow
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
)

# Create some random colors
color = np.random.randint(0, 255, (100, 3))

# Take first frame and find corners in it
ret, old_frame = cap.read()
old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
p0 = cv.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)

# Initialize canvas display
height, width = old_frame.shape[:2]
canvas = Canvas(width=width, height=height)
display(canvas)


def init_tracking():
    """Reset video to first frame and re-detect feature points."""
    global old_gray, p0, mask, color, cap
    cap.release()
    cap = cv.VideoCapture(video)
    ret, old_frame = cap.read()
    if not ret:
        return 0
    old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
    p0 = cv.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    mask = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (100, 3))
    return 1  # frame counter starts at 1


frame_count = init_tracking()

while True:
    # Handle restart
    if _restart:
        _restart = False
        frame_count = init_tracking()
        if frame_count == 0:
            print("No frames grabbed!")
            break

    ret, frame = cap.read()
    if not ret:
        print("No frames grabbed!")
        break

    frame_count += 1
    frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # calculate optical flow
    p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    # Select good points
    if p1 is not None:
        good_new = p1[st == 1]
        good_old = p0[st == 1]
    else:
        good_new = np.array([])
        good_old = np.array([])

    # draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()
        mask = cv.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
        frame = cv.circle(frame, (int(a), int(b)), 5, color[i].tolist(), -1)
    img = cv.add(frame, mask)

    # Draw frame counter + point count on the image
    text = f"Frame: {frame_count}  |  Points: {len(good_new)}"
    cv.putText(
        img,
        text,
        (10, 30),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv.LINE_AA,
    )

    # Convert BGR to RGB and add alpha channel for ipycanvas
    with hold_canvas():
        img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        rgba = np.zeros((height, width, 4), dtype=np.uint8)
        rgba[:, :, :3] = img_rgb
        rgba[:, :, 3] = 255
        canvas.put_image_data(rgba)

    # Update previous frame and points
    old_gray = frame_gray.copy()
    if len(good_new) > 0:
        p0 = good_new.reshape(-1, 1, 2)

    # Animation frequency ~30Hz
    sleep(0.03)

cap.release()

(This code doesn't check how correct are the next keypoints. So even if any feature point disappears
in image, there is a chance that optical flow finds the next point which may look close to it. So
actually for a robust tracking, corner points should be detected in particular intervals. OpenCV
samples comes up with such a sample which finds the feature points at every 5 frames. It also run a
backward-check of the optical flow points got to select only good ones. Check
samples/python/lk_track.py).

See the results we got:

![image](images/opticalflow_lk.jpg)

Dense Optical Flow in OpenCV
----------------------------

Lucas-Kanade method computes optical flow for a sparse feature set (in our example, corners detected
using Shi-Tomasi algorithm). OpenCV provides another algorithm to find the dense optical flow. It
computes the optical flow for all the points in the frame. It is based on Gunnar Farneback's
algorithm which is explained in "Two-Frame Motion Estimation Based on Polynomial Expansion" by
Gunnar Farneback in 2003.

Below sample shows how to find the dense optical flow using above algorithm. We get a 2-channel
array with optical flow vectors, \f$(u,v)\f$. We find their magnitude and direction. We color code the
result for better visualization. Direction corresponds to Hue value of the image. Magnitude
corresponds to Value plane. See the code below:


In [ ]:
from time import sleep

from ipycanvas import Canvas, hold_canvas
from IPython.display import display
from ipywidgets import Button

# Shared restart flag
_restart_dense = False


def on_restart_dense_clicked(b):
    global _restart_dense
    _restart_dense = True


# Restart button
restart_dense_btn = Button(description="Restart")
restart_dense_btn.on_click(on_restart_dense_clicked)
display(restart_dense_btn)

cap = cv.VideoCapture(video)

# Initialize canvas display
ret, frame1 = cap.read()
if not ret:
    print("No frames grabbed!")
else:
    height, width = frame1.shape[:2]
    canvas = Canvas(width=width, height=height)
    display(canvas)

    # Dense optical flow canvas to display flow as BGR image
    flow_canvas = Canvas(width=width, height=height)
    display(flow_canvas)


def init_dense_flow():
    """Reset video to first frame."""
    global prvs
    cap.release()
    cap.open(video)
    ret, frame1 = cap.read()
    if not ret:
        return False
    prvs = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    return True


if not init_dense_flow():
    print("No frames grabbed!")

frame_count = 0
hsv = np.zeros_like(frame1)
hsv[..., 1] = 255

while True:
    # Handle restart
    if _restart_dense:
        _restart_dense = False
        if not init_dense_flow():
            print("No frames grabbed!")
            break
        frame_count = 0

    ret, frame2 = cap.read()
    if not ret:
        print("No frames grabbed!")
        break

    frame_count += 1
    next_gray = cv.cvtColor(frame2, cv.COLOR_BGR2GRAY)
    flow = cv.calcOpticalFlowFarneback(prvs, next_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX)
    bgr = cv.cvtColor(hsv, cv.COLOR_HSV2BGR)

    # Draw frame counter on the flow image
    text = f"Frame: {frame_count}"
    cv.putText(
        bgr,
        text,
        (10, 30),
        cv.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv.LINE_AA,
    )

    # Display original frame and dense optical flow side by side
    with hold_canvas():
        # Original frame
        frame_rgb = cv.cvtColor(frame2, cv.COLOR_BGR2RGB)
        rgba_frame = np.zeros((height, width, 4), dtype=np.uint8)
        rgba_frame[:, :, :3] = frame_rgb
        rgba_frame[:, :, 3] = 255
        canvas.put_image_data(rgba_frame)

        # Dense optical flow
        flow_rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
        rgba_flow = np.zeros((height, width, 4), dtype=np.uint8)
        rgba_flow[:, :, :3] = flow_rgb
        rgba_flow[:, :, 3] = 255
        flow_canvas.put_image_data(rgba_flow)

    prvs = next_gray

    # Animation frequency ~30Hz
    sleep(0.03)

cap.release()